# 第 13 章 · SGEMM ①：从 naive 到 shared memory 分块

本章回答：矩阵乘法 C = A × B 最直接的 GPU 写法为什么只有峰值的 1%？只改"哪个线程算哪个元素"为什么能快 6 倍？shared memory 分块（tiling）怎么让每个数从显存读一次、用 32 次？

配套讲义：仓库里 `cuda-programming/notes/lessons/13-sgemm-tiling.md`（讲解、动画、练习都在讲义里）。对应原文：Boehm《How to Optimize a CUDA Matmul Kernel for cuBLAS-like Performance: a Worklog》Kernel 1–3（`materials/text/blog-boehm-cuda-matmul-worklog.txt` 第 20–62 行性能表、第 63–382 行）；Best Practices §10.2.3.2 Shared Memory in Matrix Multiplication（`text/cuda-c-best-practices-guide.txt` 第 1772–1936 行）；PMPP 第 5 版 ch.5（Memory architecture and data locality）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 Kernel 1 的强度 | 2 FLOP ÷ 8 字节 | `2 / 8` | 0.25，A6000 脊点 39 |
| 第二步 Kernel 1 读 A | `x = threadIdx.x` | `warp_read_sectors(x1, 0, 4096)`（第 6 章的 `sectors`） | 32 箱 |
| 第二步 Kernel 2 | `x = tx / 32`、`y = tx % 32` | `warp_read_sectors(x2, 0, …)` / `warp_read_sectors(0, y2, …)` | 读 A 1 箱（广播），读 B 4 箱（合并） |
| 第三步 每个数读几次 | $2MNK / (MK + KN)$ | `reuse_factor(4096, 4096, 4096)` | 4096 |
| 第四步 Kernel 3 | 沿 K 每段 32 | `tiled_gemm(A, B, 32)` vs `naive_gemm(A, B)` | 结果一致，显存读少 32 倍；shared 8 KB |
| 第五步 Boehm 表 | ÷ cuBLAS 23250 | `BOEHM` | 1.3%、8.5%、12.8% |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `BOEHM`**（来自 `ch13_sgemm_tiling.py`）

In [ ]:
BOEHM = {"1 naive": 309, "2 合并访存": 1987, "3 shared 分块": 2980, "cuBLAS": 23250}

**函数 `kernel1_coords`**（来自 `ch13_sgemm_tiling.py`）：Kernel 1：x（行）= blockIdx.x * 32 + threadIdx.x，y（列）= blockIdx.y * 32 + threadIdx.y。

In [ ]:
def kernel1_coords(tx: np.ndarray, bx: int = 0, by: int = 0, ty: int = 0, bs: int = 32):
    """Kernel 1：x（行）= blockIdx.x * 32 + threadIdx.x，y（列）= blockIdx.y * 32 + threadIdx.y。"""
    return bx * bs + tx, by * bs + ty

**函数 `kernel2_coords`**（来自 `ch13_sgemm_tiling.py`）：Kernel 2：x = blockIdx.x * 32 + threadIdx.x / 32，y = blockIdx.y * 32 + threadIdx.x % 32。

In [ ]:
def kernel2_coords(tx: np.ndarray, bx: int = 0, by: int = 0, bs: int = 32):
    """Kernel 2：x = blockIdx.x * 32 + threadIdx.x / 32，y = blockIdx.y * 32 + threadIdx.x % 32。"""
    return bx * bs + tx // bs, by * bs + tx % bs

**函数 `naive_gemm`**（来自 `ch13_sgemm_tiling.py`）：每个"线程"(x, y) 做一个长度 K 的点积；返回 C 和从显存读的数的个数。

In [ ]:
def naive_gemm(A: np.ndarray, B: np.ndarray) -> tuple[np.ndarray, int]:
    """每个"线程"(x, y) 做一个长度 K 的点积；返回 C 和从显存读的数的个数。"""
    M, K = A.shape
    _, N = B.shape
    C = np.zeros((M, N), dtype=A.dtype)
    reads = 0
    for x in range(M):
        for y in range(N):
            C[x, y] = A[x, :] @ B[:, y]     # 读 K 个 A、K 个 B
            reads += 2 * K
    return C, reads

**函数 `reuse_factor`**（来自 `ch13_sgemm_tiling.py`）：naive 一共读 2MNK 次，A、B 只有 MK + KN 个数：平均每个数被读几次。

In [ ]:
def reuse_factor(m: int, n: int, k: int) -> float:
    """naive 一共读 2MNK 次，A、B 只有 MK + KN 个数：平均每个数被读几次。"""
    return 2 * m * n * k / (m * k + k * n)

**函数 `tiled_gemm`**（来自 `ch13_sgemm_tiling.py`）：Kernel 3：每个 block 负责 C 的 bs × bs 小块，沿 K 每次搬 A、B 各一个 bs × bs 的 tile 进 shared。

In [ ]:
def tiled_gemm(A: np.ndarray, B: np.ndarray, bs: int = 32) -> tuple[np.ndarray, int]:
    """Kernel 3：每个 block 负责 C 的 bs × bs 小块，沿 K 每次搬 A、B 各一个 bs × bs 的 tile 进 shared。"""
    M, K = A.shape
    _, N = B.shape
    C = np.zeros((M, N), dtype=A.dtype)
    reads = 0
    for bx in range(0, M, bs):
        for by in range(0, N, bs):
            acc = np.zeros((bs, bs), dtype=A.dtype)
            for bk in range(0, K, bs):
                As = A[bx:bx + bs, bk:bk + bs]       # ① 搬进 shared（每人搬 1 个）
                Bs = B[bk:bk + bs, by:by + bs]
                reads += As.size + Bs.size
                acc += As @ Bs                        # ③ 在 shared 里做 bs 次乘加
            C[bx:bx + bs, by:by + bs] = acc
    return C, reads

**常量 `SECTOR`**（来自 `ch06_coalescing.py`）

In [ ]:
SECTOR = 32

**函数 `sectors`**（来自 `ch06_coalescing.py`）：32 个地址落在几箱：不同的 ⌊地址 / 32⌋ 的个数。

In [ ]:
def sectors(byte_addrs: np.ndarray) -> int:
    """32 个地址落在几箱：不同的 ⌊地址 / 32⌋ 的个数。"""
    return len(np.unique(np.asarray(byte_addrs) // SECTOR))

**函数 `warp_read_sectors`**（来自 `ch13_sgemm_tiling.py`）：一个 warp 读 M[row * ld + col]（float）落在几箱。

In [ ]:
def warp_read_sectors(rows, cols, ld: int) -> int:
    """一个 warp 读 M[row * ld + col]（float）落在几箱。"""
    return sectors((np.asarray(rows) * ld + np.asarray(cols)) * 4)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 Kernel 1 的算术强度：每次乘加 2 FLOP、读 8 字节")
print(f"  {2 / 8} FLOP/byte，A6000 脊点 ≈ {30e12 / 768e9:.0f}；性能 {BOEHM['1 naive'] / BOEHM['cuBLAS']:.1%} cuBLAS")

In [ ]:
print("\n第二步 一个 warp（threadIdx.x = 0..31）读 A[x * K + i]，K = 4096、i = 0")
tx = np.arange(32)
x1, _ = kernel1_coords(tx)
x2, y2 = kernel2_coords(tx)
print("  Kernel 1：32 个线程的 x =", x1[:4], "… → 读 A 落在", warp_read_sectors(x1, 0, 4096), "箱")
print("  Kernel 2：x 全是", np.unique(x2), "，读 A 落在", warp_read_sectors(x2, 0, 4096), "箱（广播）")
print("  Kernel 2：y =", y2[:4], "… 连续，读 B[i * N + y] 落在", warp_read_sectors(0, y2, 4096), "箱（合并）")
print(f"  性能 {BOEHM['2 合并访存'] / BOEHM['cuBLAS']:.1%}，比 Kernel 1 快 {BOEHM['2 合并访存'] / BOEHM['1 naive']:.1f} 倍")

In [ ]:
print("\n第三步 M = N = K = 4096：每个数平均被读几次")
print("  ", reuse_factor(4096, 4096, 4096))

In [ ]:
print("\n第四步 Kernel 3：64 × 64 的小矩阵，tile 32")
rng = np.random.default_rng(0)
A = rng.standard_normal((64, 64)).astype(np.float32)
B = rng.standard_normal((64, 64)).astype(np.float32)
C1, r1 = naive_gemm(A, B)
C3, r3 = tiled_gemm(A, B, 32)
print("  两种写法结果一致：", np.allclose(C1, A @ B, atol=1e-4) and np.allclose(C3, A @ B, atol=1e-4))
print(f"  显存读：naive {r1} 次，分块 {r3} 次，少了 {r1 // r3} 倍（= tile 边长）")
print("  每 block 的 shared：2 × 32 × 32 × 4 =", 2 * 32 * 32 * 4, "字节 = 8 KB")
print(f"  性能 {BOEHM['3 shared 分块'] / BOEHM['cuBLAS']:.1%}")

## 3. 代码里才看得到的

1. **`tiled_gemm` 里 `As @ Bs` 一行代表 block 里 1024 个线程的 32 次乘加**：NumPy 一次算完，真正的 kernel 是每个线程做 `tmp += As[ty][k] * Bs[k][tx]`，循环 32 次。
2. **显存读次数只数"从 A、B 读进来的数"**：没有模拟 L1 / L2 缓存，所以 naive 的 524288 次是最坏情况；讲义说 Kernel 2 已经有不错的 L1 命中率，这也是 Kernel 3 实测只快 1.5 倍、不是 32 倍的原因。
3. **64 × 64 是为了跑得快**：讲义的 4096 × 4096 用 Python 双重循环要跑很久，比例关系（少 32 倍）不随尺寸变。

## 4. 自己改一改

- 把 `tiled_gemm` 的 tile 换成 16、8，看显存读少几倍。
- 把 `warp_read_sectors(x1, 0, 4096)` 的 K 换成 4、8，看 K 很小时 Kernel 1 读 A 还要几箱。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 一人一格做点积：每算 2 次读 8 字节，1% 峰值；
> threadIdx.x 对列：A 广播、B 合并，快 6 倍；
> 一个数被读几千次，搬进 shared 用 32 次；
> 搬、等、算、等：写完要等，读完再覆盖也要等；
> 瓶颈从显存挪到 shared，下一步搬进寄存器。